<a href="https://colab.research.google.com/github/aydanali/ECON3916-Stats-and-ML/blob/main/Lab13/StatsML_Lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

In [3]:
# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/aydanali/ECON3916-Stats-and-ML/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [7]:
naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        20:02:16   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [9]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:02:33   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [11]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

In [12]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

In [13]:
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price,Price_Residuals,Age_Residuals
0,77.5,38.1,684100.56,-56823.332740,0.621803
1,11.0,95.1,413634.22,19000.990249,-13.689856
2,47.7,73.5,456709.35,-69149.815200,3.233510
3,61.9,60.3,624533.95,18481.157582,5.347789
4,100.8,16.4,870137.54,-2619.815668,4.053610
...,...,...,...,...,...
995,87.7,10.1,932592.35,21560.763160,-14.814575
996,21.2,91.8,412741.12,-1940.516556,-6.511286
997,96.5,14.5,880901.56,-3398.817768,-1.986001
998,20.1,95.1,396659.79,2026.560249,-4.589856


In [14]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802138


In [15]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 1. Simulate the dataset (replace with your actual df) ──────────────────────
'''
np.random.seed(42)
n = 1000
Property_Age         = np.random.uniform(5, 110, n)
Distance_to_Tech_Hub = np.random.uniform(5, 100, n)
Sale_Price           = (
    500_000
    + 4_200  * Property_Age
    - 3_800  * Distance_to_Tech_Hub
    + np.random.normal(0, 40_000, n)
)
df = pd.DataFrame({
    "Property_Age":          Property_Age,
    "Distance_to_Tech_Hub":  Distance_to_Tech_Hub,
    "Sale_Price":             Sale_Price,
})
'''
# ── 2. Fit the OLS model ───────────────────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
# sm.add_constant prepends a column of 1s so statsmodels can estimate the intercept.
# The resulting design matrix has columns: [const, Property_Age, Distance_to_Tech_Hub]

model  = sm.OLS(df["Sale_Price"], X)
result = model.fit()
print(result.summary())

# ── 3. Extract coefficients from the fitted result ─────────────────────────────
# result.params is a pandas Series indexed by variable name.
#   result.params["const"]                → β₀ (intercept)
#   result.params["Property_Age"]         → β₁
#   result.params["Distance_to_Tech_Hub"] → β₂
b0 = result.params["const"]                # intercept
b1 = result.params["Property_Age"]         # slope for Property_Age
b2 = result.params["Distance_to_Tech_Hub"] # slope for Distance_to_Tech_Hub

print(f"\nExtracted coefficients:\n  β₀ (intercept)  = {b0:,.2f}")
print(f"  β₁ (Age)        = {b1:,.2f}")
print(f"  β₂ (Distance)   = {b2:,.2f}")

# ── 4. Build the meshgrid for the regression surface ─────────────────────────
# np.linspace produces evenly-spaced values spanning the observed range of each
# predictor.  We use 40 points per axis → 40×40 = 1,600 grid nodes, which is
# dense enough for a smooth surface without becoming slow to render.

age_range  = np.linspace(df["Property_Age"].min(),
                          df["Property_Age"].max(), 40)          # shape (40,)
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                          df["Distance_to_Tech_Hub"].max(), 40)  # shape (40,)

# np.meshgrid turns two 1-D arrays into two 2-D matrices where every combination
# of (age, dist) is represented.  age_grid[i, j] and dist_grid[i, j] together
# give the coordinates of grid node (i, j).
age_grid, dist_grid = np.meshgrid(age_range, dist_range)
# Both grids: shape (40, 40)

# Apply the OLS equation  Ŷ = β₀ + β₁·Age + β₂·Distance  across the entire grid.
# Because age_grid and dist_grid are NumPy arrays, this is vectorised — no loops.
price_grid = b0 + b1 * age_grid + b2 * dist_grid  # shape (40, 40)

# ── 5. Build the Plotly figure ─────────────────────────────────────────────────
residuals = df["Sale_Price"] - result.fittedvalues  # raw residuals

fig = go.Figure()

# — Scatter: actual data points, coloured by residual sign ——————————————————
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=3,
        color=residuals,          # continuous colour scale tied to residual value
        colorscale="RdBu",        # red = under-predicted, blue = over-predicted
        cmin=-residuals.abs().max(),
        cmax= residuals.abs().max(),
        colorbar=dict(title="Residual ($)", x=1.05),
        opacity=0.75,
    ),
    name="Observed data",
    hovertemplate=(
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Sale Price: $%{z:,.0f}<extra></extra>"
    ),
))

# — Surface: the fitted regression hyperplane ——————————————————————————————
# x, y, z must be 2-D arrays of the same shape — exactly what meshgrid gave us.
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_grid,
    colorscale="Blues",
    opacity=0.45,
    showscale=False,
    name="OLS hyperplane",
    hovertemplate=(
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Predicted: $%{z:,.0f}<extra></extra>"
    ),
))

# ── 6. Layout ──────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            f"3D OLS Regression Hyperplane<br>"
            f"<sup>Ŷ = {b0:,.0f} + {b1:,.0f}·Age + {b2:,.0f}·Distance</sup>"
        ),
        x=0.5,
    ),
    scene=dict(
        xaxis=dict(title="Property Age (yrs)"),
        yaxis=dict(title="Distance to Tech Hub (km)"),
        zaxis=dict(title="Sale Price ($)", tickformat="$,.0f"),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.8)),
    ),
    margin=dict(l=0, r=120, t=100, b=0),
    legend=dict(x=0, y=1),
    height=700,
)

fig.show()
# fig.write_html("regression_3d_hyperplane.html")  # ← uncomment to save

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:13:22   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 1.203e+06 